In [98]:
# We have already covered RAG. We have used Langchain to build it.
# Now we look at Langgraph.

In [99]:
# What's different about Langgraph is that it is a graph-based framework for building language models. 
# It allows us to create a graph of nodes and edges, 
# where each node represents a piece of information and each edge represents a relationship between pieces of information. 

In [100]:
# What if there were different flows in which the coversation could take place?
# What if there were multiple pieces of code that could be executed based on some conditions?

# In such cases, we use Langgraph.

In [101]:
# We will explore the following concepts in steps.
# This will probably give us all the exploration ideas we need to deploy application to Pilot phase.

**Structure:**
1. State — TypedDict, Annotated reducers, MessagesState
2. Nodes and edges — basic graph construction and execution
3. Conditional edges — routing and branching
4. Tools — defining tools, ToolNode, single and multiple tools
5. ReAct agent — LLM + tools in a loop
6. Checkpointers — persistence across graph runs
7. Human-in-the-loop — interrupt, inspect, resume
8. Subgraphs — composing graphs within graphs
9. Multi-agent — supervisor pattern, handoff pattern
10. Streaming, Send API, map-reduce

In [102]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir)))

In [103]:
from dotenv import load_dotenv
load_dotenv()

True

---
#### Section 1 — State

State is the single most important concept in LangGraph. 
Every node reads from state and writes back to state. The graph itself is just a set of rules for which node runs next — all the actual data lives in state.

**Three things to understand:**
1. State is a `TypedDict` — it defines the schema of data flowing through the graph
2. Reducers control how a field is updated when a node writes to it — overwrite vs. append
3. `MessagesState` is a built-in state type for LLM conversations that handles message list accumulation automatically

In [104]:
# To make the nodes and edges more compliant with our required structures,
# I always recommend my teams to follow structures like this:

# from typing import TypedDict,Any, Dict, List, Optional
# class Structure(TypedDict):
#     name: str
#     description: Optional[str]
#     input_schema: Optional[Dict[str, Any]]
#     output_schema: Optional[Dict[str, Any]]

In [105]:
# 1a — Plain TypedDict state (overwrite semantics)
# When a node returns {"count": 5}, the field is overwritten with 5.

from typing import TypedDict

class SimpleState(TypedDict):
    user_message: str
    response: str
    count: int

# Any node that returns {"count": 5} will set count = 5, not count += 5.
# This is plain overwrite — the default behaviour.

In [106]:
# 1b — Reducers: controlling how fields merge
# When you want a field to ACCUMULATE rather than overwrite, use Annotated + a reducer function.

import operator
from typing import Annotated
from typing import TypedDict

class AccumulatingState(TypedDict):
    # Plain overwrite
    current_query: str

    # Append — operator.add on a list means extend, not replace
    # If node A returns {"steps": ["step1"]} and node B returns {"steps": ["step2"]},
    # the final state has steps = ["step1", "step2"]
    steps: Annotated[list, operator.add]

    # You can use any function as a reducer
    # This one keeps only the maximum value
    best_score: Annotated[float, lambda old, new: max(old, new) if old is not None else new]


# WHY THIS MATTERS:
# In a multi-agent graph where several nodes all want to add tool results or
# reasoning steps, you NEED append semantics or every node overwrites the previous one's work.
# This is one of the most common sources of bugs in LangGraph code.
# I'll explain it it the next few sections.

In [107]:
# 1c — MessagesState: the built-in state for LLM conversation graphs
# This is a pre-built TypedDict with a single field: messages
# The reducer is add_messages — it appends new messages but DEDUPLICATES by message ID.
# That deduplication is what makes it safe to reuse messages across graph runs.

In [108]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# MessagesState is equivalent to:
# class MessagesState(TypedDict):
#     messages: Annotated[list[AnyMessage], add_messages]

# You can extend it to add our own fields
class RAGState(MessagesState):
    # Inherits: messages: Annotated[list, add_messages]
    # Add our own fields:
    session_id: str
    rag_context: str
    retrieved_chunks: Annotated[list, operator.add]  # accumulate across retrieval nodes
    query_intent: str  # overwrite — only one intent at a time


In [109]:
# 1d — Understanding add_messages deduplication
# This is subtle but important: if you send a message with the same ID twice,
# add_messages replaces the old one rather than duplicating it.
# This is how LangGraph handles tool result updates without creating duplicate messages.

In [110]:
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

In [111]:
existing = [HumanMessage(content="hello", id="msg-1")]
update   = [HumanMessage(content="hello updated", id="msg-1")]  # same ID

In [112]:
result = add_messages(existing, update)
print(f"Messages after merge: {len(result)} message(s)")
print(f"Content: {result[0].content}")
# Output: 1 message, content = "hello updated"
# It replaced, not duplicated

Messages after merge: 1 message(s)
Content: hello updated


In [113]:
# Compare with appending a new message (different ID)
new_msg = [HumanMessage(content="new message", id="msg-2")]
result2 = add_messages(existing, new_msg)
print(f"Messages after append: {len(result2)} message(s)")

Messages after append: 2 message(s)


---
#### Section 2 — Nodes and Edges

A node is any Python function (sync or async) that takes state as input and returns a dict of state updates.

An edge is a directed connection between two nodes. The graph runs by following edges from the entry point until it reaches `END`.

In [114]:
# 2a — Minimal graph: two nodes, one edge

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class State(TypedDict):
    value: str

In [115]:
# Nodes are plain functions — they receive the FULL current state
# and return a dict of ONLY the fields they want to update.
# Fields not returned are left unchanged.

def node_a(state: State) -> dict:
    print(f"node_a received: {state['value']}")
    return {"value": state["value"] + " -> A"}

def node_b(state: State) -> dict:
    print(f"node_b received: {state['value']}")
    return {"value": state["value"] + " -> B"}

In [116]:
# Build the graph
builder = StateGraph(State)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

In [117]:
# Wire edges: START -> node_a -> node_b -> END
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

In [118]:
graph = builder.compile()

In [119]:
# Run it
result = graph.invoke({"value": "start"})
print(f"Final state: {result}")

node_a received: start
node_b received: start -> A
Final state: {'value': 'start -> A -> B'}


In [121]:
# This is the most basic graph you can build. It has two nodes that modify a string in sequence.

In [122]:
# 2b — Nodes can be async — this is important for our FastAPI application

import asyncio
import nest_asyncio
nest_asyncio.apply()  # allows nested event loops in Jupyter
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [123]:
class State(TypedDict):
    text: str

In [124]:
async def async_node(state: State) -> dict:
    # Simulating an async DB call or LLM call
    await asyncio.sleep(0)  # yield to event loop
    return {"text": state["text"] + " [processed async]"}

In [125]:
builder = StateGraph(State)
builder.add_node("process", async_node)
builder.add_edge(START, "process")
builder.add_edge("process", END)

In [126]:
graph = builder.compile()

In [127]:
# ainvoke for async graphs
result = await graph.ainvoke({"text": "hello"})
print(result)

{'text': 'hello [processed async]'}


In [128]:
# 2c — Parallel edges: two nodes running at the same time
# When multiple edges leave the same node, LangGraph runs the targets IN PARALLEL
# and waits for all to finish before proceeding. Their state updates are merged
# using the reducers defined on the state.

In [129]:
import operator
from typing import Annotated
from langgraph.graph import StateGraph, START, END

In [130]:
class State(TypedDict):
    input: str
    results: Annotated[list, operator.add]  # MUST be append — both nodes write to this


In [131]:
def worker_a(state: State) -> dict:
    return {"results": [f"A processed: {state['input']}"]}

def worker_b(state: State) -> dict:
    return {"results": [f"B processed: {state['input']}"]}

def aggregator(state: State) -> dict:
    print(f"Aggregating {len(state['results'])} results")
    return {"results": state["results"]}  # pass through

In [132]:
builder = StateGraph(State)
builder.add_node("worker_a", worker_a)
builder.add_node("worker_b", worker_b)
builder.add_node("aggregator", aggregator)

builder.add_edge(START, "worker_a")    # both edges from START
builder.add_edge(START, "worker_b")    # → parallel execution
builder.add_edge("worker_a", "aggregator")
builder.add_edge("worker_b", "aggregator")
builder.add_edge("aggregator", END)

In [133]:
graph = builder.compile()
result = graph.invoke({"input": "test", "results": []})
print(result["results"])
# Both worker_a and worker_b results are in the list

Aggregating 2 results
['A processed: test', 'B processed: test', 'A processed: test', 'B processed: test']


---
#### Section 3 — Conditional Edges

Conditional edges let a node decide at runtime which node to go to next. This is how you implement routing, retry logic, and loops.

In [134]:
# 3a — Basic conditional edge
# add_conditional_edges takes: source_node, routing_function
# The routing function receives the current state and returns a string — the name of the next node.


In [135]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [136]:
class State(TypedDict):
    query: str
    mode: str  # "fast" or "deep"
    response: str

In [137]:
def classify_query(state: State) -> dict:
    # In reality this would call an LLM or run heuristics
    word_count = len(state["query"].split())
    mode = "deep" if word_count > 5 else "fast"
    return {"mode": mode}

def fast_answer(state: State) -> dict:
    return {"response": f"[FAST] Quick answer to: {state['query']}"}

def deep_answer(state: State) -> dict:
    return {"response": f"[DEEP] Thorough answer to: {state['query']}"}

In [138]:
# The routing function — returns the name of the NEXT node
def route_by_mode(state: State) -> str:
    return state["mode"]  # returns "fast" or "deep"

In [139]:
builder = StateGraph(State)
builder.add_node("classify", classify_query)
builder.add_node("fast", fast_answer)
builder.add_node("deep", deep_answer)

builder.add_edge(START, "classify")

# Conditional edge from "classify":
# call route_by_mode(state) — if it returns "fast" go to fast node, else go to deep node
builder.add_conditional_edges(
    "classify",
    route_by_mode,
    # Optional explicit mapping — makes routing logic readable
    {"fast": "fast", "deep": "deep"}
)
builder.add_edge("fast", END)
builder.add_edge("deep", END)

In [140]:
graph = builder.compile()

In [141]:
r1 = graph.invoke({"query": "hi", "mode": "", "response": ""})
print(r1["response"])

[FAST] Quick answer to: hi


In [142]:
r2 = graph.invoke({"query": "can you explain the methodology in detail", "mode": "", "response": ""})
print(r2["response"])

[DEEP] Thorough answer to: can you explain the methodology in detail


In [143]:
# 3b — Conditional edge routing to END
# You can route directly to END from a conditional edge — useful for early exit.

In [144]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [145]:
class State(TypedDict):
    message: str
    blocked: bool
    response: str

In [146]:
def safety_check(state: State) -> dict:
    is_harmful = "hack" in state["message"].lower()
    return {"blocked": is_harmful}

def generate_response(state: State) -> dict:
    return {"response": f"Response to: {state['message']}"}

def route_after_safety(state: State) -> str:
    if state["blocked"]:
        return END  # skip generation entirely
    return "generate"

In [147]:
builder = StateGraph(State)
builder.add_node("safety", safety_check)
builder.add_node("generate", generate_response)

builder.add_edge(START, "safety")
builder.add_conditional_edges("safety", route_after_safety)
builder.add_edge("generate", END)

In [148]:
graph = builder.compile()

r1 = graph.invoke({"message": "tell me about AI", "blocked": False, "response": ""})
print(f"Not blocked: {r1['response']}")

Not blocked: Response to: tell me about AI


In [149]:
r2 = graph.invoke({"message": "how to hack systems", "blocked": False, "response": ""})
print(f"Blocked: response='{r2['response']}', blocked={r2['blocked']}")

Blocked: response='', blocked=True


In [152]:
# 3c — Loop with conditional exit: the retry / refine pattern
# A graph can loop back to a previous node. The conditional edge decides whether to
# loop again or exit. This is the foundation of any iterative refinement workflow.

# For example, you might have a validation step that checks if the output is correct or not.
# This is how you can setup a limit so that the code does not go into infinite loop.

In [153]:
import operator
from typing import Annotated
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    query: str
    attempts: Annotated[list, operator.add]
    final_answer: str
    quality_score: float

In [154]:
def generate(state: State) -> dict:
    attempt_num = len(state["attempts"]) + 1
    answer = f"Answer attempt #{attempt_num} for: {state['query']}"
    # Simulate improving quality each attempt
    score = min(0.5 * attempt_num, 1.0)
    return {
        "attempts": [answer],
        "final_answer": answer,
        "quality_score": score,
    }

def evaluate(state: State) -> str:
    if state["quality_score"] >= 1.0:
        return "done"
    if len(state["attempts"]) >= 3:  # hard cap to prevent infinite loop
        return "done"
    return "generate"  # loop back

In [155]:
builder = StateGraph(State)
builder.add_node("generate", generate)

builder.add_edge(START, "generate")
builder.add_conditional_edges(
    "generate",
    evaluate,
    {"generate": "generate", "done": END}  # loop back or exit
)

In [156]:
graph = builder.compile()
result = graph.invoke({"query": "explain quantum computing", "attempts": [], "final_answer": "", "quality_score": 0.0})
print(f"Total attempts: {len(result['attempts'])}")
print(f"Final score: {result['quality_score']}")
print(f"Final answer: {result['final_answer']}")

Total attempts: 2
Final score: 1.0
Final answer: Answer attempt #2 for: explain quantum computing


---
#### Section 4 — Tools

Tools are functions that an LLM can choose to call. In LangGraph, `ToolNode` is a pre-built node that executes whatever tools the LLM requested in its last message.

In [157]:
# 4a — Defining tools with @tool decorator
# The docstring becomes the tool description the LLM sees.
# Type annotations become the parameter schema.

In [159]:
# As we go along we will use some custom functions as tools, tools from libraries and much more.
# For now let's see how to define a simple tool with the @tool decorator.

In [160]:
from langchain_core.tools import tool

In [161]:
@tool
def search_documents(query: str, top_k: int = 5) -> str:
    """Search the uploaded research documents for information relevant to the query.
    
    Args:
        query: The search query string.
        top_k: Number of results to return.
    """
    # In the real app this would call PgVectorRetrievalRepository.search()
    return f"[Mock] Found {top_k} chunks for query: '{query}'"

In [162]:
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Example: '2 + 2 * 3'"""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

In [163]:
@tool
def get_paper_metadata(filename: str) -> str:
    """Retrieve metadata for an uploaded research paper by filename."""
    return f"[Mock] Metadata for {filename}: title='Sample Paper', authors=['Author A'], year=2024"


In [164]:
# Inspect what the LLM sees
print(search_documents.name)
print(search_documents.description)
print(search_documents.args)

search_documents
Search the uploaded research documents for information relevant to the query.

    Args:
        query: The search query string.
        top_k: Number of results to return.
{'query': {'title': 'Query', 'type': 'string'}, 'top_k': {'default': 5, 'title': 'Top K', 'type': 'integer'}}


In [165]:
# 4b — ToolNode: the pre-built node that executes tool calls
# ToolNode looks at the last AIMessage in state.messages, finds any tool_calls,
# executes them, and appends ToolMessage results back into state.messages.

In [166]:
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage

In [167]:
tools = [search_documents, calculate, get_paper_metadata]
tool_node = ToolNode(tools)

In [168]:
# Simulate what ToolNode receives: an AIMessage with tool_calls
# (In a real graph the LLM produces this — here we construct it manually to see the mechanics)
fake_ai_message = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "search_documents",
            "args": {"query": "transformer attention mechanism", "top_k": 3},
            "id": "call_001",
            "type": "tool_call",
        }
    ]
)

In [169]:
fake_ai_message

AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'search_documents', 'args': {'query': 'transformer attention mechanism', 'top_k': 3}, 'id': 'call_001', 'type': 'tool_call'}], invalid_tool_calls=[])

In [170]:
# 4b — ToolNode: the pre-built node that executes tool calls
# ToolNode looks at the last AIMessage in state.messages, finds any tool_calls,
# executes them, and appends ToolMessage results back into state.messages.

In [171]:
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage
from langgraph.runtime import Runtime
from langgraph._internal._runnable import CONFIG_KEY_RUNTIME
from langchain_core.runnables import RunnableConfig

In [172]:
tools = [search_documents, calculate, get_paper_metadata]
tool_node = ToolNode(tools)

In [173]:
# Simulate what ToolNode receives: an AIMessage with tool_calls
# (In a real graph the LLM produces this — here we construct it manually to see the mechanics)
fake_ai_message = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "search_documents",
            "args": {"query": "transformer attention mechanism", "top_k": 3},
            "id": "call_001",
            "type": "tool_call",
        }
    ]
)

In [174]:
# LangGraph's recent upgrades require a Runtime in config when invoking ToolNode outside a graph
runtime_config = RunnableConfig(configurable={CONFIG_KEY_RUNTIME: Runtime()})
state = {"messages": [fake_ai_message]}
result = tool_node.invoke(state, config=runtime_config)

print(f"ToolNode returned {len(result['messages'])} new message(s)")
print(f"Type: {type(result['messages'][0]).__name__}")
print(f"Content: {result['messages'][0].content}")

ToolNode returned 1 new message(s)
Type: ToolMessage
Content: [Mock] Found 3 chunks for query: 'transformer attention mechanism'


In [175]:
# 4c — Multiple parallel tool calls in one LLM response
# Modern LLMs can request multiple tools at once. ToolNode handles this by
# running them in parallel and returning one ToolMessage per call.

In [176]:
fake_parallel_calls = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "search_documents",
            "args": {"query": "attention mechanism"},
            "id": "call_001",
            "type": "tool_call",
        },
        {
            "name": "calculate",
            "args": {"expression": "2 ** 10"},
            "id": "call_002",
            "type": "tool_call",
        },
        {
            "name": "get_paper_metadata",
            "args": {"filename": "paper.pdf"},
            "id": "call_003",
            "type": "tool_call",
        },
    ]
)

In [177]:
state = {"messages": [fake_parallel_calls]}
result = tool_node.invoke(state, config=runtime_config)

print(f"ToolNode returned {len(result['messages'])} message(s) (one per tool call)")
for msg in result["messages"]:
    print(f"  tool_call_id={msg.tool_call_id}: {msg.content[:60]}")

ToolNode returned 3 message(s) (one per tool call)
  tool_call_id=call_001: [Mock] Found 5 chunks for query: 'attention mechanism'
  tool_call_id=call_002: 1024
  tool_call_id=call_003: [Mock] Metadata for paper.pdf: title='Sample Paper', authors


In [178]:
# 4d — tools_condition: the pre-built routing function for tool use
# tools_condition looks at the last message:
# - if it has tool_calls → return "tools" (go to ToolNode)
# - if it has no tool_calls → return END (LLM is done)

from langgraph.prebuilt import tools_condition
from langchain_core.messages import AIMessage

# With tool calls
state_with_calls = {"messages": [fake_parallel_calls]}
print(tools_condition(state_with_calls))  # "tools"

# Without tool calls
state_without_calls = {"messages": [AIMessage(content="Here is our answer.")]}
print(tools_condition(state_without_calls))  # END

tools
__end__


---
#### Section 5 — ReAct Agent

ReAct (Reason + Act) is the loop: LLM thinks → calls a tool → observes result → thinks again → calls another tool or answers. It's the foundation of all agents.

In [179]:
# 5a — ReAct agent built from scratch (so you understand what the prebuilt does)
# The structure is always:
#   llm_node → (conditional: has tool calls?) → tool_node → llm_node → ...

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

In [180]:
@tool
def search_documents(query: str) -> str:
    """Search uploaded research papers for the given query."""
    # Mock — replace with real retrieval
    return f"Found relevant content about '{query}': Transformers use self-attention to model long-range dependencies."

@tool
def get_paper_count() -> int:
    """Return the number of papers uploaded in this session."""
    return 3  # mock

tools = [search_documents, get_paper_count]

In [185]:
# Bind tools to the LLM — this tells the LLM which tools it can call
# This was how agentic approach started everywhere.
# Let the LLM access the tools.


llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

In [182]:
def call_llm(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)

In [183]:
builder = StateGraph(MessagesState)
builder.add_node("llm", call_llm)
builder.add_node("tools", tool_node)

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")  # after tools, always go back to LLM

agent = builder.compile()

In [184]:
# Run the agent
from langchain_core.messages import HumanMessage

result = await agent.ainvoke({
    "messages": [HumanMessage(content="How many papers do I have? Also search for info about attention mechanisms.")]
})

# Print the conversation
for msg in result["messages"]:
    role = type(msg).__name__
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    print(f"[{role}]: {content[:120]}")
    print()

[HumanMessage]: How many papers do I have? Also search for info about attention mechanisms.

[AIMessage]: 

[ToolMessage]: 3

[ToolMessage]: Found relevant content about 'attention mechanisms': Transformers use self-attention to model long-range dependencies.

[AIMessage]: You have 3 papers uploaded. Regarding attention mechanisms, I found that transformers use self-attention to model long-r



In [186]:
# After devs and researchers found this to be a useful structure,
# Agents were developed along with reasoning capabilitys, tool use, and more.

In [187]:
# 5b — Same agent using the prebuilt create_agent
# Now that you understand what it does under the hood, the prebuilt is fine to use.
# create_agent is exactly the graph you built above, plus optional system prompt support.

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


In [189]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a research paper assistant. Use the available tools to answer questions about uploaded documents."
)

In [190]:
result = await agent.ainvoke({
    "messages": [HumanMessage(content="Search for information about self-attention.")]
})

# Final response only
print(result["messages"][-1].content)

Self-attention is used in Transformers to model long-range dependencies. If you want, I can provide more detailed information or specific aspects related to self-attention.


In [191]:
# 5c — Limiting the ReAct loop: max iterations
# Unbounded agents can loop forever and run up costs.
# The cleanest way to cap iterations is a counter in state + conditional exit.

In [192]:
import operator
from typing import Annotated
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

In [193]:
class AgentState(MessagesState):
    tool_call_count: int  # plain overwrite — we set this explicitly

In [194]:
MAX_TOOL_CALLS = 5

In [195]:
def call_llm(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def route_with_limit(state: AgentState) -> str:
    # Standard tools_condition check
    last = state["messages"][-1]
    if not getattr(last, "tool_calls", None):
        return END
    # Enforce cap
    if state.get("tool_call_count", 0) >= MAX_TOOL_CALLS:
        return END
    return "tools"

def tool_node_with_count(state: AgentState) -> dict:
    # Execute tools then increment counter
    tool_results = ToolNode(tools).invoke(state)
    return {
        **tool_results,
        "tool_call_count": state.get("tool_call_count", 0) + 1,
    }

In [196]:
builder = StateGraph(AgentState)
builder.add_node("llm", call_llm)
builder.add_node("tools", tool_node_with_count)

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", route_with_limit)
builder.add_edge("tools", "llm")

bounded_agent = builder.compile()

In [197]:
result = await bounded_agent.ainvoke({
    "messages": [HumanMessage(content="Search for attention, then search for transformers, then search for BERT.")],
    "tool_call_count": 0,
})

In [198]:
print(f"Tool calls used: {result['tool_call_count']}")
print(f"Final response: {result['messages'][-1].content[:200]}")

Tool calls used: 1
Final response: I found relevant content about all three queries: attention, transformers, and BERT. The common relevant information is that transformers use self-attention to model long-range dependencies. If you wa


---
#### Section 6 — Checkpointers and Persistence

A checkpointer saves the graph state after every node execution to a storage backend. This enables:
- Resuming a graph run after it was interrupted
- Multi-turn conversations where each call resumes where the last left off
- Human-in-the-loop (Section 7 depends on this)

Each run is identified by a `thread_id`. Multiple turns in the same conversation share the same `thread_id`.

In [ ]:
# 6a — MemorySaver: in-process checkpointer for development
# State is stored in a Python dict in memory. Lost on process restart.
# In production you would swap this for AsyncSqliteSaver or AsyncPostgresSaver.

from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [200]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
checkpointer = MemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=checkpointer,
)

In [201]:
# thread_id links all turns in the same conversation
config = {"configurable": {"thread_id": "session-abc-123"}}

In [202]:
# Turn 1
r1 = await agent.ainvoke(
    {"messages": [HumanMessage(content="My name is Alice.")]},
    config=config
)
print("Turn 1:", r1["messages"][-1].content)

Turn 1: Hello Alice! How can I assist you today?


In [203]:
# Turn 2 — the agent REMEMBERS the conversation without you passing history
r2 = await agent.ainvoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config
)
print("Turn 2:", r2["messages"][-1].content)

Turn 2: Your name is Alice. How can I assist you further, Alice?


In [204]:
# 6b — Inspecting state between turns
# get_state(config) returns the current persisted state for a thread

state_snapshot = agent.get_state(config)
print(f"Number of messages in checkpointed state: {len(state_snapshot.values['messages'])}")
print(f"Next nodes to run: {state_snapshot.next}")

# get_state_history returns every checkpoint saved for this thread
history = list(agent.get_state_history(config))
print(f"Number of checkpoints: {len(history)}")

Number of messages in checkpointed state: 4
Next nodes to run: ()
Number of checkpoints: 6


In [205]:
# 6c — Separate threads = separate conversations
# Different thread_ids are completely isolated from each other

config_user_b = {"configurable": {"thread_id": "session-xyz-456"}}

r3 = await agent.ainvoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config_user_b
)
print("User B (no prior context):", r3["messages"][-1].content)
# The agent will say it doesn't know the name — no cross-thread leakage

User B (no prior context): I don't have access to your name based on the current conversation. Could you please tell me your name?


In [206]:
# 6d — Understanding the relationship between LangGraph checkpointers and our PostgreSQL memory
# our APP currently stores messages manually in poc2prod.chats.
# LangGraph checkpointers would store the ENTIRE graph state (messages + all other state fields).
#
# Two integration strategies:
#
# Strategy A — Replace: use LangGraph's AsyncPostgresSaver as the checkpointer.
#   - The checkpointer stores messages in its own schema.
#   - our poc2prod.chats table is no longer needed.
#   - Simpler, but you lose custom columns like embeddings, sender classification, etc.
#
# Strategy B — Coexist: use MemorySaver (or no checkpointer) for the graph,
#   and continue writing to poc2prod.chats manually at the API layer.
#   - our existing schema is unchanged.
#   - You reconstruct the messages list from the DB at the start of each graph invocation.
#   - More code but full control.
#
# FOR our APP: Strategy B is the right choice because you need the embeddings column
# on chats for long-term memory retrieval, which LangGraph's checkpointer doesn't provide.


---
#### Section 7 — Human-in-the-Loop

Interrupting a graph mid-execution to wait for human input, then resuming. This requires a checkpointer — the graph saves state before the interrupt and restores it when resumed.

In [207]:
# 7a — interrupt_before: pause BEFORE a node runs, inspect state, then decide
# The graph runs up to the interrupt point, saves state, and returns.
# You resume it by calling invoke/ainvoke again with the same config.

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage, AIMessage

In [208]:
class State(MessagesState):
    plan: str
    approved: bool

def plan_response(state: State) -> dict:
    plan = f"Plan: I will search the documents for: '{state['messages'][-1].content}' and synthesize a response."
    return {"plan": plan}

def execute_plan(state: State) -> dict:
    response = f"Executed: {state['plan']}"
    return {"messages": [AIMessage(content=response)]}

In [209]:
checkpointer = MemorySaver()

builder = StateGraph(State)
builder.add_node("plan", plan_response)
builder.add_node("execute", execute_plan)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)

In [210]:
# interrupt_before="execute" means: run up to and including "plan",
# then PAUSE before running "execute"
graph = builder.compile(checkpointer=checkpointer, interrupt_before=["execute"])

In [211]:
config = {"configurable": {"thread_id": "hitl-001"}}

In [212]:
# First invocation — runs "plan", then pauses
r1 = await graph.ainvoke(
    {"messages": [HumanMessage(content="What is self-attention?")], "plan": "", "approved": False},
    config=config
)

# Inspect the plan before execution
current_state = graph.get_state(config)
print(f"Graph paused at: {current_state.next}")
print(f"Plan: {current_state.values['plan']}")
print("Awaiting approval...")

Graph paused at: ('execute',)
Plan: Plan: I will search the documents for: 'What is self-attention?' and synthesize a response.
Awaiting approval...


In [80]:
# 7b — Resume by calling invoke again with the same config
# Pass None as input when resuming — the graph reads state from the checkpointer

print("Human approved. Resuming...")

r2 = await graph.ainvoke(None, config=config)

print(f"Final response: {r2['messages'][-1].content}")

Human approved. Resuming...
Final response: Executed: Plan: I will search the documents for: 'What is self-attention?' and synthesize a response.


In [213]:
# 7c — Update state before resuming: the human changes the plan
# update_state lets you inject new values into the checkpointed state
# before the graph continues

In [214]:
config2 = {"configurable": {"thread_id": "hitl-002"}}

In [215]:
# Run to the interrupt
await graph.ainvoke(
    {"messages": [HumanMessage(content="Explain BERT.")], "plan": "", "approved": False},
    config=config2
)

state_before = graph.get_state(config2)
print(f"Original plan: {state_before.values['plan']}")

# Human edits the plan
graph.update_state(
    config2,
    {"plan": "MODIFIED PLAN: Search for BERT specifically, focus on pre-training details."},
)

state_after = graph.get_state(config2)
print(f"Modified plan: {state_after.values['plan']}")

# Resume with the modified plan
r3 = await graph.ainvoke(None, config=config2)
print(f"Response with modified plan: {r3['messages'][-1].content}")

Original plan: Plan: I will search the documents for: 'Explain BERT.' and synthesize a response.
Modified plan: MODIFIED PLAN: Search for BERT specifically, focus on pre-training details.
Response with modified plan: Executed: MODIFIED PLAN: Search for BERT specifically, focus on pre-training details.


In [216]:
# 7d — interrupt() inside a node: dynamic interrupts
# Instead of declaring interrupt points at compile time, you can call interrupt()
# from inside any node. This is more flexible — the node decides at runtime
# whether to pause based on conditions.

from langgraph.types import interrupt, Command
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

In [217]:
class State(TypedDict):
    query: str
    response: str
    needs_confirmation: bool

In [218]:
def generate_and_maybe_pause(state: State) -> dict:
    # Complex query — ask human to confirm before we proceed
    if len(state["query"].split()) > 8:
        # interrupt() saves state and raises — the graph halts here
        # The value passed to interrupt() is returned to the caller
        human_decision = interrupt({
            "question": "This is a complex query. Should I use deep retrieval mode?",
            "query": state["query"]
        })
        # When resumed, human_decision contains whatever the human passed back
        use_deep = human_decision.get("use_deep", False)
        response = f"[{'DEEP' if use_deep else 'FAST'}] Answer for: {state['query']}"
    else:
        response = f"[FAST] Quick answer for: {state['query']}"
    return {"response": response}


In [219]:
checkpointer = MemorySaver()
builder = StateGraph(State)
builder.add_node("generate", generate_and_maybe_pause)
builder.add_edge(START, "generate")
builder.add_edge("generate", END)

graph = builder.compile(checkpointer=checkpointer)
config = {"configurable": {"thread_id": "dynamic-interrupt-001"}}

In [220]:
# Long query — will trigger interrupt
result = await graph.ainvoke(
    {"query": "explain the differences between BERT GPT and T5 architectures in detail", "response": "", "needs_confirmation": False},
    config=config
)
print(f"Graph state after interrupt: {result}")

state = graph.get_state(config)
print(f"Interrupted at: {state.next}")
# result contains the interrupt payload
print(f"Interrupt payload: {state.tasks[0].interrupts[0].value if state.tasks else 'N/A'}")

Graph state after interrupt: {'query': 'explain the differences between BERT GPT and T5 architectures in detail', 'response': '', 'needs_confirmation': False, '__interrupt__': [Interrupt(value={'question': 'This is a complex query. Should I use deep retrieval mode?', 'query': 'explain the differences between BERT GPT and T5 architectures in detail'}, id='9d80d4f3bcd0ee381e207b6a1381e299')]}
Interrupted at: ('generate',)
Interrupt payload: {'question': 'This is a complex query. Should I use deep retrieval mode?', 'query': 'explain the differences between BERT GPT and T5 architectures in detail'}


In [221]:
# 7e — Resume an interrupt() with Command
# Pass Command(resume=<value>) as input to send data back to the interrupted node

from langgraph.types import Command

result = await graph.ainvoke(
    Command(resume={"use_deep": True}),  # human says: yes, use deep mode
    config=config
)
print(f"Final response: {result['response']}")

Final response: [DEEP] Answer for: explain the differences between BERT GPT and T5 architectures in detail


---
#### Section 8 — Subgraphs

A subgraph is a compiled graph used as a node inside another graph. This lets you decompose complex graphs into reusable components. our fast mode and deep mode are natural subgraphs.

In [222]:
# 8a — Basic subgraph
# A subgraph is just a compiled StateGraph passed as the node function.
# State schemas do NOT need to be identical — they just need to share
# the keys that are passed between the parent and subgraph.

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [223]:
# --- Subgraph A: fast retrieval ---
class FastState(TypedDict):
    query: str
    context: str   # output passed back to parent

def fast_retrieve(state: FastState) -> dict:
    return {"context": f"[1 chunk] Fast result for: {state['query']}"}

fast_builder = StateGraph(FastState)
fast_builder.add_node("retrieve", fast_retrieve)
fast_builder.add_edge(START, "retrieve")
fast_builder.add_edge("retrieve", END)
fast_subgraph = fast_builder.compile()


In [224]:
# --- Subgraph B: deep retrieval ---
class DeepState(TypedDict):
    query: str
    context: str

def deep_retrieve(state: DeepState) -> dict:
    return {"context": f"[5 chunks + re-ranking] Deep result for: {state['query']}"}

def deep_synthesize(state: DeepState) -> dict:
    return {"context": state["context"] + " [synthesized]"}

deep_builder = StateGraph(DeepState)
deep_builder.add_node("retrieve", deep_retrieve)
deep_builder.add_node("synthesize", deep_synthesize)
deep_builder.add_edge(START, "retrieve")
deep_builder.add_edge("retrieve", "synthesize")
deep_builder.add_edge("synthesize", END)
deep_subgraph = deep_builder.compile()


In [225]:
# --- Parent graph: routes to one of the two subgraphs ---
class ParentState(TypedDict):
    query: str
    mode: str
    context: str
    response: str

def route_mode(state: ParentState) -> str:
    return state["mode"]

def generate_final(state: ParentState) -> dict:
    return {"response": f"Answer (using context): {state['context']}"}

parent_builder = StateGraph(ParentState)
# Subgraphs are added as nodes — the node name is arbitrary
parent_builder.add_node("fast_mode", fast_subgraph)
parent_builder.add_node("deep_mode", deep_subgraph)
parent_builder.add_node("generate", generate_final)

parent_builder.add_conditional_edges(
    START,
    route_mode,
    {"fast": "fast_mode", "deep": "deep_mode"}
)
parent_builder.add_edge("fast_mode", "generate")
parent_builder.add_edge("deep_mode", "generate")
parent_builder.add_edge("generate", END)

parent_graph = parent_builder.compile()

In [226]:
# Run in fast mode
r1 = parent_graph.invoke({"query": "what is attention?", "mode": "fast", "context": "", "response": ""})
print("Fast:", r1["response"])

Fast: Answer (using context): [1 chunk] Fast result for: what is attention?


In [227]:
# Run in deep mode
r2 = parent_graph.invoke({"query": "compare BERT and GPT in detail", "mode": "deep", "context": "", "response": ""})
print("Deep:", r2["response"])

Deep: Answer (using context): [5 chunks + re-ranking] Deep result for: compare BERT and GPT in detail [synthesized]


In [85]:
# 8b — State key sharing between parent and subgraph
# LangGraph automatically handles passing shared keys between parent and subgraph.
# Keys that exist in both schemas are passed IN when the subgraph starts,
# and updated values are passed OUT when the subgraph ends.
# Keys that only exist in the subgraph are local to it.

# Rule: the subgraph schema must be a SUBSET of (or compatible with) the parent schema
# for the shared keys. In the example above:
#   ParentState has: query, mode, context, response
#   FastState has:   query, context
# LangGraph passes query + context IN, and gets context back OUT. mode and response
# are invisible to the subgraph — this is intentional isolation.


---
#### Section 9 — Multi-Agent Systems

Two standard patterns for coordinating multiple agents:
1. **Supervisor pattern**: a supervisor agent decides which worker agent to call next
2. **Handoff pattern**: agents pass control directly to each other

Each pattern suits different use cases.

In [228]:
# 9a — Supervisor pattern
# A supervisor LLM sees the task and decides which specialized worker to run.
# After each worker completes, control returns to the supervisor.
# The supervisor can call workers multiple times, in any order.

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, Literal
import operator
from typing import Annotated

In [229]:
# Worker agents — each is a simple function (could also be a subgraph)
@tool
def retrieval_worker_tool(query: str) -> str:
    """Search documents for relevant information about the query."""
    return f"[RETRIEVAL] Found context: 'Transformers use multi-head attention with Q, K, V matrices for query=\'{query}\''"

@tool
def citation_worker_tool(claim: str) -> str:
    """Find citations and references that support or contradict a claim."""
    return f"[CITATION] Found 3 papers supporting: '{claim}' — Vaswani 2017, Devlin 2019, Brown 2020"

@tool
def summary_worker_tool(text: str) -> str:
    """Summarize a long piece of text into key bullet points."""
    return f"[SUMMARY] Key points: • Attention is core • Parallelizable • Scales well | from: '{text[:50]}...'"

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

supervisor_tools = [retrieval_worker_tool, citation_worker_tool, summary_worker_tool]
supervisor_llm = llm.bind_tools(supervisor_tools)

In [230]:
from langgraph.prebuilt import ToolNode, tools_condition

def supervisor_node(state: MessagesState) -> dict:
    system = """You are a research supervisor. Use the available tools to help answer the user's question:
    - retrieval_worker_tool: search documents for information
    - citation_worker_tool: find supporting citations  
    - summary_worker_tool: summarize retrieved information
    Use multiple tools as needed to give a complete, well-cited answer."""
    from langchain_core.messages import SystemMessage
    messages = [SystemMessage(content=system)] + state["messages"]
    response = supervisor_llm.invoke(messages)
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("tools", ToolNode(supervisor_tools))

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", tools_condition)
builder.add_edge("tools", "supervisor")

supervisor_graph = builder.compile()

In [231]:
result = await supervisor_graph.ainvoke({
    "messages": [HumanMessage(content="Explain transformer attention mechanisms and find citations.")]
})

print(result["messages"][-1].content)

Transformer attention mechanisms are a core component of the Transformer architecture. They use multi-head attention, which involves three matrices: query (Q), key (K), and value (V). The attention mechanism computes a weighted sum of the values, where the weights are determined by how compatible the queries are with the corresponding keys. Multi-head attention allows the model to attend to information from different representation subspaces at different positions simultaneously, which helps the model capture various aspects of the input data effectively. This mechanism is parallelizable and scales well, making it efficient for processing sequences.

Key citations supporting this explanation include:
- Vaswani et al., 2017 (the original Transformer paper)
- Devlin et al., 2019 (BERT)
- Brown et al., 2020 (GPT-3)

If you want, I can provide more detailed explanations or excerpts from these papers.


In [232]:
# 9b — Handoff pattern: agents pass control to each other directly
# Each agent can decide to hand off to a specific other agent.
# No central supervisor — agents coordinate peer-to-peer.
# Better when the workflow is linear: A -> B -> C

from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, Annotated
import operator


In [233]:
class PipelineState(MessagesState):
    retrieved_context: str
    citations: str
    final_answer: str
    current_agent: str

def retrieval_agent(state: PipelineState) -> dict:
    query = state["messages"][-1].content
    context = f"Retrieved: transformer paper content for '{query}'"
    return {
        "retrieved_context": context,
        "current_agent": "citation",  # hand off to citation agent
    }

def citation_agent(state: PipelineState) -> dict:
    citations = f"Citations for context: Vaswani et al. 2017"
    return {
        "citations": citations,
        "current_agent": "synthesis",  # hand off to synthesis agent
    }

def synthesis_agent(state: PipelineState) -> dict:
    answer = (
        f"Based on retrieved context and {state['citations']}, "
        f"here is the answer: {state['retrieved_context']}"
    )
    return {"final_answer": answer, "current_agent": "done"}


In [234]:
def route_by_current_agent(state: PipelineState) -> str:
    return state["current_agent"]

In [235]:
builder = StateGraph(PipelineState)
builder.add_node("retrieval", retrieval_agent)
builder.add_node("citation", citation_agent)
builder.add_node("synthesis", synthesis_agent)

builder.add_edge(START, "retrieval")
# Retrieval hands off based on current_agent field
builder.add_conditional_edges("retrieval", route_by_current_agent, {"citation": "citation"})
builder.add_conditional_edges("citation", route_by_current_agent, {"synthesis": "synthesis"})
builder.add_conditional_edges("synthesis", route_by_current_agent, {"done": END})

pipeline = builder.compile()

In [236]:
result = pipeline.invoke({
    "messages": [HumanMessage(content="Explain self-attention")],
    "retrieved_context": "",
    "citations": "",
    "final_answer": "",
    "current_agent": "",
})
print(result["final_answer"])

Based on retrieved context and Citations for context: Vaswani et al. 2017, here is the answer: Retrieved: transformer paper content for 'Explain self-attention'


In [237]:
# 9c — When to use supervisor vs handoff
#
# SUPERVISOR:
#   - Non-deterministic flow: you don't know upfront which agents will be needed
#   - Flexible ordering: retrieval might run before OR after citation check
#   - Cost: every step requires an LLM call to the supervisor
#   - Good for: open-ended question answering, complex reasoning tasks
#
# HANDOFF:
#   - Deterministic pipeline: you know the order upfront
#   - Low overhead: no supervisor LLM calls between steps
#   - Good for: document processing pipelines, structured workflows
#
# FOR OUR APP:
#   - Fast mode → handoff pattern: retrieve → generate (two steps, fixed order, cheap)
#   - Deep mode → supervisor pattern: supervisor decides how many retrieval passes,
#     whether to run citation check, whether to decompose the query into sub-queries


---
#### Section 10 — Streaming, Send API, and Map-Reduce

Three advanced patterns for production use:
- **Streaming**: emit tokens or events as they happen instead of waiting for the full result
- **Send API**: dynamically spawn multiple graph runs in parallel with different inputs
- **Map-reduce**: fan out to N parallel workers, collect and aggregate all results

In [238]:
# 10a — Streaming: two modes
# astream() with stream_mode="updates" — emits after each node completes
# astream() with stream_mode="messages" — emits LLM tokens as they generate

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0, streaming=True)
agent = create_agent(model=llm, tools=tools)

print("--- stream_mode='updates': emits after each node ---")
async for event in agent.astream(
    {"messages": [HumanMessage(content="How many papers do I have?")]},
    stream_mode="updates",
):
    # event is a dict: {node_name: state_update}
    node_name = list(event.keys())[0]
    print(f"  Node '{node_name}' completed")

--- stream_mode='updates': emits after each node ---
  Node 'model' completed
  Node 'tools' completed
  Node 'model' completed


In [239]:
# 10b — Token-level streaming from an LLM node
# stream_mode="messages" emits (message_chunk, metadata) tuples as LLM generates tokens

print("--- stream_mode='messages': LLM tokens as they arrive ---")
print("Response: ", end="", flush=True)

async for chunk, metadata in agent.astream(
    {"messages": [HumanMessage(content="Briefly explain attention mechanisms.")]},
    stream_mode="messages",
):
    # metadata["langgraph_node"] tells you which node is generating
    if metadata.get("langgraph_node") == "agent" and chunk.content:
        print(chunk.content, end="", flush=True)

print()  # newline after streaming

--- stream_mode='messages': LLM tokens as they arrive ---
Response: 


In [240]:
# 10c — Streaming into our existing SSE endpoint
# This is the integration pattern for our FastAPI /stream endpoint.
# our current stream_message endpoint uses provider.astream_chat().
# When LangGraph runs the agent, you replace that with graph.astream().

import json

async def graph_stream_to_sse(graph, user_message: str, thread_id: str):
    """Adapter: converts LangGraph astream output into SSE events.
    
    This replaces the event_generator() function in src/api/chat.py
    once LangGraph is wired in.
    """
    config = {"configurable": {"thread_id": thread_id}}

    async for chunk, metadata in graph.astream(
        {"messages": [HumanMessage(content=user_message)]},
        config=config,
        stream_mode="messages",
    ):
        node = metadata.get("langgraph_node", "")

        # Tool call started — tell the frontend a tool is running
        if hasattr(chunk, "tool_calls") and chunk.tool_calls:
            for tc in chunk.tool_calls:
                yield f"data: {json.dumps({'type': 'tool_start', 'tool': tc['name']})}\n\n"

        # LLM token from the agent node
        elif node == "agent" and chunk.content:
            yield f"data: {json.dumps({'type': 'token', 'content': chunk.content})}\n\n"

    yield f"data: {json.dumps({'type': 'done'})}\n\n"

# Test it
async for event in graph_stream_to_sse(agent, "Search for attention mechanisms.", "thread-001"):
    print(event.strip())

data: {"type": "tool_start", "tool": "search_documents"}
data: {"type": "tool_start", "tool": ""}
data: {"type": "done"}


In [241]:
# 10d — Send API: dynamic fan-out
# Send lets you spawn the SAME node multiple times in parallel with DIFFERENT inputs.
# This is different from parallel edges (which go to different nodes).
# Use case: decompose a complex query into N sub-queries, process each in parallel.

import operator
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

In [242]:
class OverallState(TypedDict):
    original_query: str
    sub_queries: list[str]
    results: Annotated[list, operator.add]  # MUST be append reducer
    final_answer: str

class SubQueryState(TypedDict):
    sub_query: str
    result: str

def decompose_query(state: OverallState) -> dict:
    # In deep mode, break the complex query into focused sub-queries
    query = state["original_query"]
    sub_queries = [
        f"What is {query.split()[0]}?",
        f"How does {query.split()[0]} work?",
        f"What are the applications of {query.split()[0]}?",
    ]
    return {"sub_queries": sub_queries}

def process_sub_query(state: SubQueryState) -> dict:
    # This runs once per sub-query, in parallel
    result = f"Result for '{state['sub_query']}': [mock retrieved content]"
    return {"results": [result]}

def aggregate_results(state: OverallState) -> dict:
    combined = "\n".join(state["results"])
    return {"final_answer": f"Comprehensive answer from {len(state['results'])} sub-queries:\n{combined}"}


In [243]:
# The Send function: after decompose, spawn one process_sub_query per sub-query
def fan_out(state: OverallState) -> list[Send]:
    return [
        Send("process_sub_query", {"sub_query": sq})
        for sq in state["sub_queries"]
    ]


In [244]:
builder = StateGraph(OverallState)
builder.add_node("decompose", decompose_query)
builder.add_node("process_sub_query", process_sub_query)
builder.add_node("aggregate", aggregate_results)

builder.add_edge(START, "decompose")
# fan_out returns a list of Send objects — LangGraph runs them all in parallel
builder.add_conditional_edges("decompose", fan_out)
builder.add_edge("process_sub_query", "aggregate")
builder.add_edge("aggregate", END)

fan_out_graph = builder.compile()

In [245]:
result = fan_out_graph.invoke({
    "original_query": "transformer attention mechanisms",
    "sub_queries": [],
    "results": [],
    "final_answer": "",
})

print(f"Sub-queries processed: {len(result['results'])}")
print(result["final_answer"])

Sub-queries processed: 3
Comprehensive answer from 3 sub-queries:
Result for 'What is transformer?': [mock retrieved content]
Result for 'How does transformer work?': [mock retrieved content]
Result for 'What are the applications of transformer?': [mock retrieved content]


In [246]:
# 10e — Full map-reduce pattern
# Map: fan out with Send to process N items in parallel
# Reduce: aggregate node collects all results via append reducer
#
# IMPORTANT: the aggregate node runs ONCE per invocation of the Send target,
# not once per batch. LangGraph waits for ALL parallel branches to complete
# before moving on — this is the implicit reduce step.
#
# In the graph above, aggregate runs AFTER all process_sub_query branches finish.
# The Annotated[list, operator.add] reducer on 'results' accumulates all outputs.

# For deep mode in our app, the map-reduce pattern looks like:
#
# decompose_query → [Send(retrieve, {sub_query: q1}),
#                    Send(retrieve, {sub_query: q2}),
#                    Send(retrieve, {sub_query: q3})]
#        ↓ (all run in parallel)
# retrieve node × 3  (each does a vector search)
#        ↓ (all collected via operator.add reducer)
# rerank_and_synthesize  (dedup + rerank combined chunks, pass to LLM)
#        ↓
# generate_final_answer

In [247]:
# 10f — SUMMARY: How everything connects to our fast/deep mode architecture

summary = """
FAST MODE graph:
    START
      → embed_query          (embed user message — reuse existing code)
      → retrieve             (single vector search — top 5 chunks, our existing retrieval repo)
      → generate             (LLM with RAG context injected into system prompt)
      → END
    
    Pattern: handoff, no tools, no loops
    Latency: 1 embed + 1 DB query + 1 LLM call

DEEP MODE graph:
    START
      → analyze_query         (LLM classifies intent, decomposes into sub-queries)
      → [Send × N]            (fan out N sub-queries to retrieve node in parallel)
      → retrieve × N          (each sub-query does an independent vector search)
      → rerank_and_deduplicate (collapse N result sets, remove duplicate chunks)
      → generate              (LLM with rich context — may loop back if quality check fails)
      → [loop back OR end]    (conditional: quality check passes → END, fails → retry generate)
      → END

    Pattern: supervisor / map-reduce + loop
    Latency: 1 embed + N DB queries (parallel) + 1 rerank + 1-3 LLM calls

TOP-LEVEL ROUTER:
    START
      → route_by_user_setting  (conditional edge: fast → fast_subgraph, deep → deep_subgraph)
      → [fast_subgraph | deep_subgraph]
      → persist_to_db          (write messages + embeddings to poc2prod.chats)
      → END

    The persist_to_db node replaces our current repo.add_message() calls.
    You keep our existing PostgreSQL schema — LangGraph runs the reasoning,
    our DB handles persistence.
"""

print(summary)


FAST MODE graph:
    START
      → embed_query          (embed user message — reuse existing code)
      → retrieve             (single vector search — top 5 chunks, our existing retrieval repo)
      → generate             (LLM with RAG context injected into system prompt)
      → END

    Pattern: handoff, no tools, no loops
    Latency: 1 embed + 1 DB query + 1 LLM call

DEEP MODE graph:
    START
      → analyze_query         (LLM classifies intent, decomposes into sub-queries)
      → [Send × N]            (fan out N sub-queries to retrieve node in parallel)
      → retrieve × N          (each sub-query does an independent vector search)
      → rerank_and_deduplicate (collapse N result sets, remove duplicate chunks)
      → generate              (LLM with rich context — may loop back if quality check fails)
      → [loop back OR end]    (conditional: quality check passes → END, fails → retry generate)
      → END

    Pattern: supervisor / map-reduce + loop
    Latency: 1 embed 

---
### What we can build next (in order)

Now that we have understood all the primitives, here is the recommended build sequence for integrating LangGraph into our app:

**Step 1 — Wire in a basic LangGraph pipeline (no agents)**
Replace `ChatService.get_response_async()` with a simple graph: `embed → retrieve → generate`. Verify the output is identical to our current behaviour. No behaviour change — just architecture change.

**Step 2 — Add the fast/deep mode router**
Add a `mode` field to state. Add a conditional edge at the start that routes to the fast subgraph or deep subgraph. Implement fast subgraph first (it is trivial — same as step 1). Keep deep subgraph as a stub that just calls fast for now.

**Step 3 — Implement deep subgraph with multi-query fan-out**
Use the Send API to decompose the query and run parallel retrievals. Add the rerank + deduplicate node. Wire into the existing `PgVectorRetrievalRepository`.

**Step 4 — Add LLM-as-judge quality loop in deep mode**
After generation, add a quality check node that loops back to generate if the answer is insufficient. Cap at 3 iterations.

**Step 5 — Add streaming**
Replace `provider.astream_chat()` in our SSE endpoint with `graph.astream(stream_mode='messages')`. Add `tool_start` SSE events so the frontend can show "Searching documents..."

**Step 6 — Add human-in-the-loop for deep mode**
Optionally: interrupt before execution to show the user the decomposed sub-queries and let them edit before retrieval runs.